# Strength Scan Overview

## Configuration

In [ ]:
from pathlib import Path
import os

SCAN_ROOT = Path(
    os.environ.get(
        "WWGPT_STRENGTH_SCAN_ROOT",
        "/tmp/wwpgd_strength_scan",
    )
)

In [ ]:
import sys

for repo_src in ((Path.cwd() / "src").resolve(), (Path.cwd().parent / "src").resolve()):
    if repo_src.exists() and str(repo_src) not in sys.path:
        sys.path.insert(0, str(repo_src))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from wwgpt.strength_scan_analysis import resolve_scan_root, analyze_strength_scan
try:
    scan_root = resolve_scan_root(SCAN_ROOT)
except FileNotFoundError:
    scan_root = resolve_scan_root(Path.cwd().parent / "tests/fixtures/strength_scan")
analysis_dir = analyze_strength_scan(scan_root)
fig_dir = scan_root / "analysis" / "notebook_overview"
fig_dir.mkdir(parents=True, exist_ok=True)
print(scan_root)

## Scan manifest

## Run inventory

## Completion and stability audit

## Pairing audit

## Terminal validation-loss comparison

## Validation-loss trajectories

## Projection magnitude

## Throughput and runtime

## Strength-response summary

## Preliminary interpretation

In [ ]:
import json

manifest = json.loads((scan_root / "scan_manifest.json").read_text())
runs = pd.read_csv(analysis_dir / "strength_scan_runs.csv")
terminal = pd.read_csv(analysis_dir / "strength_scan_terminal.csv")
proj = pd.read_csv(analysis_dir / "strength_scan_projection.csv")
summary = pd.read_csv(analysis_dir / "strength_scan_summary.csv")

display(manifest)
display(runs)
display(runs[runs.complete == True])
display(runs[(runs.stable == False) | (runs.complete == False)])
display(terminal)
display(summary)

if summary.seed_count.max() <= 1:
    print("One-seed scan: confidence intervals are unavailable.")

## Plots

The notebook now renders every overview plot inline and saves the same images under `analysis/notebook_overview/`.

Accuracy plots aggregate every available scan run and visualize seed-to-seed variation with standard-deviation error bars and translucent Bollinger-style bands.


In [ ]:
saved_plots = []

def save_and_show(fig, filename):
    path = fig_dir / filename
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    saved_plots.append(path)
    display(Image(filename=str(path)))

def empty_plot(message, filename):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.text(0.5, 0.5, message, ha="center", va="center", wrap=True)
    ax.axis("off")
    save_and_show(fig, filename)

def plot_strength_line(df, x, y, filename, title, ylabel=None, xlabel="strength"):
    if df.empty or y not in df or x not in df:
        empty_plot(f"No data available for {title}.", filename)
        return
    plot_df = df[[x, y]].dropna().sort_values(x)
    if plot_df.empty:
        empty_plot(f"No finite data available for {title}.", filename)
        return
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(plot_df[x], plot_df[y], marker="o")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--") if "diff" in filename else None
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel or y)
    ax.grid(True, alpha=0.25)
    save_and_show(fig, filename)

In [ ]:
plot_strength_line(
    terminal,
    "strength",
    "wwpgd_minus_adamw_final_loss",
    "final_loss_diff.png",
    "WW-PGD minus AdamW final validation loss by strength",
    "final validation-loss difference",
)
plot_strength_line(
    terminal,
    "strength",
    "wwpgd_minus_adamw_minimum_loss",
    "min_loss_diff.png",
    "WW-PGD minus AdamW minimum validation loss by strength",
    "minimum validation-loss difference",
)

## Training and test accuracy across strengths

Training and test (validation) accuracy plots summarize all WW-PGD strength scans and AdamW controls. Error bars and shaded bands show variation across seeds.


In [ ]:
accuracy_metrics = [
    ("train_top1_accuracy", "Training top-1 accuracy"),
    ("val_top1_accuracy", "Test/validation top-1 accuracy"),
    ("train_top5_accuracy", "Training top-5 accuracy"),
    ("val_top5_accuracy", "Test/validation top-5 accuracy"),
]


def collect_terminal_accuracy(runs_df):
    rows = []
    for _, r in runs_df.iterrows():
        mpath = Path(r.run_dir) / "metrics.csv"
        if not mpath.exists():
            continue
        metrics = pd.read_csv(mpath)
        if metrics.empty:
            continue
        last = metrics.iloc[-1]
        for column, label in accuracy_metrics:
            if column in metrics.columns and pd.notna(last.get(column)):
                rows.append(
                    {
                        "seed": r.seed,
                        "strength": r.strength,
                        "optimizer_family": r.optimizer_family,
                        "metric": column,
                        "metric_label": label,
                        "accuracy": float(last[column]),
                    }
                )
    return pd.DataFrame(rows)


def collect_accuracy_trajectories(runs_df):
    rows = []
    for _, r in runs_df.iterrows():
        mpath = Path(r.run_dir) / "metrics.csv"
        if not mpath.exists():
            continue
        metrics = pd.read_csv(mpath)
        if "tokens_processed" not in metrics.columns:
            continue
        for column, label in accuracy_metrics:
            if column not in metrics.columns:
                continue
            subset = metrics[["tokens_processed", column]].dropna()
            for _, m in subset.iterrows():
                rows.append(
                    {
                        "seed": r.seed,
                        "strength": r.strength,
                        "optimizer_family": r.optimizer_family,
                        "tokens_processed": m.tokens_processed,
                        "metric": column,
                        "metric_label": label,
                        "accuracy": float(m[column]),
                    }
                )
    return pd.DataFrame(rows)


terminal_accuracy = collect_terminal_accuracy(runs)
accuracy_trajectories = collect_accuracy_trajectories(runs)

def plot_terminal_accuracy(metric, filename, title):
    if terminal_accuracy.empty or metric not in set(terminal_accuracy.metric):
        empty_plot(f"No terminal accuracy data available for {title}.", filename)
        return

    fig, ax = plt.subplots(figsize=(9, 5))
    metric_df = terminal_accuracy[terminal_accuracy.metric == metric].copy()
    ww = metric_df[metric_df.optimizer_family == "wwpgd"].dropna(subset=["strength", "accuracy"])
    if not ww.empty:
        agg = (
            ww.groupby("strength")
            .agg(mean=("accuracy", "mean"), std=("accuracy", "std"), seed_count=("seed", "nunique"))
            .reset_index()
            .sort_values("strength")
        )
        agg["std"] = agg["std"].fillna(0.0)
        ax.errorbar(
            agg["strength"],
            agg["mean"],
            yerr=agg["std"],
            marker="o",
            capsize=4,
            linewidth=2,
            label="WW-PGD mean ± seed std",
            color="tab:blue",
        )
        ax.fill_between(
            agg["strength"],
            agg["mean"] - agg["std"],
            agg["mean"] + agg["std"],
            color="tab:blue",
            alpha=0.16,
            label="WW-PGD seed band",
        )

    controls = metric_df[metric_df.optimizer_family == "adamw"].dropna(subset=["accuracy"])
    if not controls.empty:
        mean = controls["accuracy"].mean()
        std = controls["accuracy"].std(ddof=1) if controls["seed"].nunique() > 1 else 0.0
        strengths = ww["strength"].dropna().sort_values().unique()
        if len(strengths):
            ax.axhline(mean, color="tab:orange", linewidth=2, label="AdamW control mean")
            ax.fill_between(
                strengths,
                mean - std,
                mean + std,
                color="tab:orange",
                alpha=0.16,
                label="AdamW seed band",
            )
        else:
            ax.errorbar(["AdamW"], [mean], yerr=[std], marker="o", capsize=4, color="tab:orange", label="AdamW mean ± seed std")

    ax.set_title(title)
    ax.set_xlabel("strength")
    ax.set_ylabel("accuracy")
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)
    save_and_show(fig, filename)


plot_terminal_accuracy("train_top1_accuracy", "train_top1_accuracy_by_strength.png", "Terminal training top-1 accuracy by strength")
plot_terminal_accuracy("val_top1_accuracy", "test_top1_accuracy_by_strength.png", "Terminal test/validation top-1 accuracy by strength")
plot_terminal_accuracy("train_top5_accuracy", "train_top5_accuracy_by_strength.png", "Terminal training top-5 accuracy by strength")
plot_terminal_accuracy("val_top5_accuracy", "test_top5_accuracy_by_strength.png", "Terminal test/validation top-5 accuracy by strength")



In [ ]:
if len(proj):
    projection_by_strength = proj.groupby("strength", as_index=False).agg(
        median_relative_frobenius_change=("median_relative_frobenius_change", "mean"),
        total_projection_runtime=("total_projection_runtime", "mean"),
    )
else:
    projection_by_strength = pd.DataFrame()

plot_strength_line(
    projection_by_strength,
    "strength",
    "median_relative_frobenius_change",
    "projection_norm.png",
    "Mean projection magnitude by strength",
    "relative Frobenius change",
)
plot_strength_line(
    projection_by_strength,
    "strength",
    "total_projection_runtime",
    "projection_runtime.png",
    "Mean projection runtime by strength",
    "projection runtime (seconds)",
)

In [ ]:
wwpgd_runs = runs[runs.optimizer_family == "wwpgd"].copy()
if not wwpgd_runs.empty:
    runtime_by_strength = wwpgd_runs.groupby("strength", as_index=False).agg(
        total_immediate_weightwatcher_runtime=("total_immediate_weightwatcher_runtime", "mean"),
        tokens_per_second=("tokens_per_second", "mean"),
        stable=("stable", "mean"),
    )
else:
    runtime_by_strength = pd.DataFrame()

plot_strength_line(
    runtime_by_strength,
    "strength",
    "total_immediate_weightwatcher_runtime",
    "immediate_ww.png",
    "Mean immediate WeightWatcher runtime by strength",
    "runtime (seconds)",
)
plot_strength_line(
    runtime_by_strength,
    "strength",
    "tokens_per_second",
    "tokens_per_second.png",
    "Mean training throughput by strength",
    "tokens per second",
)

if runtime_by_strength.empty:
    empty_plot("No WW-PGD runs are available for stability plotting.", "stability.png")
else:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(runtime_by_strength["strength"].astype(str), runtime_by_strength["stable"])
    ax.set_title("Stable-run fraction by strength")
    ax.set_xlabel("strength")
    ax.set_ylabel("stable fraction")
    ax.set_ylim(0, 1.05)
    ax.grid(True, axis="y", alpha=0.25)
    save_and_show(fig, "stability.png")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
plotted = False
for _, r in runs.iterrows():
    mpath = Path(r.run_dir) / "metrics.csv"
    if not mpath.exists():
        continue
    metrics = pd.read_csv(mpath)
    if {"tokens_processed", "val_loss"}.issubset(metrics.columns):
        label = f"{r.optimizer_family} strength={r.strength} seed={r.seed}"
        ax.plot(metrics.tokens_processed, metrics.val_loss, marker="o", linewidth=1.5, alpha=0.75, label=label)
        plotted = True

if plotted:
    ax.set_title("Validation-loss trajectories")
    ax.set_xlabel("tokens processed")
    ax.set_ylabel("validation loss")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8, ncol=2)
    save_and_show(fig, "val_trajectories.png")
else:
    plt.close(fig)
    empty_plot("No metrics.csv files with validation-loss trajectories were found.", "val_trajectories.png")

In [ ]:
plot_manifest = pd.DataFrame({"plot_file": [str(path.relative_to(scan_root)) for path in saved_plots]})
plot_manifest.to_csv(fig_dir / "plot_manifest.csv", index=False)
display(plot_manifest)
print("Observed trade-offs are shown above; no optimal strength is selected automatically.")